# 學生學習與作業版｜任務 9：從分析問題到預測問題

**姓名：**＿＿＿何韋仁＿　**學號／組別：**＿＿C114118243＿＿　**完成日期：**＿＿11/20＿＿

**學習目標：** 區分描述、探索及預測，明確定義時間點、標籤、使用情境與限制。

本單元與教師版「任務 9」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「9.2 建立教學標籤」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [ ]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


## 9.1 三種問題
- 描述：平均分數多少？
- 探索：讀書時間與分數有何關係？
- 預測：在小考前，哪些學生可能低於 75 分？

**觀念說明：**分類模型輸出類別，並不解釋造成成績的原因。本例假設行為欄位在考前取得，現有資料無時間戳記，不能驗證。


In [ ]:
# 1. 為了確保預測的準確度，使用已填補並清洗完畢的 df_clean 數據
df_predict = df.copy()
# 填補睡眠缺失值與限制螢幕時間異常值
df_predict["sleep_hours"] = df_predict["sleep_hours"].fillna(df_predict["sleep_hours"].median())
df_predict["screen_time_hours"] = df_predict["screen_time_hours"].clip(1.0, 10.0)

# 2. 進行預測：定義考前預警名單 (quiz_score 實際或預測低於 75 分的學生)
warning_students = df_predict[df_predict["quiz_score"] < 75]

print(f"--- 考前預警：預估成績低於 75 分的學生名單 (共 {len(warning_students)} 人) ---")
display(warning_students[["student_id", "department", "study_hours", "screen_time_hours", "attendance_rate", "quiz_score"]])

# 3. 輸出基本描述統計
print("預警學生的平均讀書時數：", warning_students["study_hours"].mean().round(1), "小時")
print("全體學生的平均讀書時數：", df_predict["study_hours"].mean().round(1), "小時")

## 📝 學生作業 09-Q1｜AI 查詢與作答：問題類型
**任務：**描述、探索與分類預測有何差異？各舉一個本資料可用問題。

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用學生學習資料各舉一個描述、探索及分類預測問題，並解釋資料限制。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 09-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具＿＿Colab Copilot (Gemini)
實際提示詞：我是 Python 與資料分析初學者。請用我們目前這份 60 筆學生學習作息與小考分數資料，分別舉出一個「描述性問題」、「探索性問題」及「預測性問題」的例子。並用繁體中文、台灣常用的數據分析術語，分步向我解釋這三者的核心差異、它們在此資料集所需要的欄位，以及資料的限制。＿＿
- AI 回答重點（請自行整理）：＿＿＿1.描述性問題 (Descriptive Question)：專注於「過去與現狀發生了什麼事」，使用摘要統計（如平均數、中位數、標準差）來呈現資料。例如：學生的平均讀書時數與不及格率是多少？
2.探索性問題 (Exploratory Question)：專注於「尋找變數之間的關聯性或潛在模式」，常用散佈圖或相關係數。例如：讀書時間長短、螢幕使用時間與小考分數之間是否存在某種關聯趨勢？
3.預測性問題 (Predictive Question)：專注於「利用已知特徵去預測未知的目標數值或分類」。例如：能否利用考前的出席率與讀書時數，預測哪些學生有高機率不及格（分數 < 60 或預警分數 < 75）？
4.資料限制：模擬數據不具備真實因果關係，且資料品質問題（缺失值、異常值）會大幅干擾統計結果（如 15 小時螢幕時間對相關係數的拉抬效應）。＿
- 我的理解與具體例子：＿1.描述性問題：
具體問題：我們班上這 60 位學生，各科系的人數分佈（樣本數）為何？平均出席率是多少？
所需欄位：department, attendance_rate
分析方法：利用 value_counts() 與 mean() 進行次數與平均數統計。
2.探索性問題：
具體問題：每天玩手機的螢幕使用時間愈高，是否與小考成績呈現負相關？
所需欄位：screen_time_hours, quiz_score
分析方法：使用 plt.scatter() 繪製散佈圖，並計算 Pearson 相關係數（發現排除異常值後呈顯著負相關 -0.509）。
3.預測（分類）問題：
具體問題：哪些學生因為讀書時數過低（如低於班級平均），在考前會被系統列入成績低於 75 分的「預警輔導名單」？
所需欄位：study_hours, quiz_score
分析方法：篩選 df_predict[df_predict["quiz_score"] < 75] 並比較其讀書時數與全班平均（預警學生 1.8 小時 vs 全班平均 2.5 小時）。＿＿＿
- 查證來源／方式及日期（無法查證請註明）：＿＿＿台灣教育部大專校院資訊公開平台定義、Pandas 官方說明文件（.corr() 與 .groupby() 的運算邏輯）。＿
- 我檢查或修正 AI 回答的地方：＿AI 最初提供的預測問題範例使用複迴歸預測精確分數，但我將其修正為更符合我們課堂「考前預警分類（低於 75 分）」的實務分類問題，這樣更能直接利用我們在 cell 100d4e53 中跑出來的 14 位預警名單與平均時數（1.8小時）來做實證支持，使報告與程式輸出完全吻合。＿＿＿


## 9.2 建立教學標籤
設定 `quiz_score < 75` 為 1，其他為 0；75 分本身是 0。本標籤僅供教學，不可據此判定真實學生是否需要輔導。


## 💻 學生作業 09-P1｜Vibe Coding 程式填空
**對應教師版：**任務 9「9.2 建立教學標籤」的示範程式。

**任務：**依 quiz_score 產生整數標籤 needs_support：小於 75 是 1，75 或以上是 0。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：依 quiz_score 產生整數標籤 needs_support：小於 75 是 1，75 或以上是 0。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [ ]:
# 建立副本，再用條件產生布林值，astype(int) 轉成 1／0
labeled = df.copy()

# [學生填空] 將 quiz_score 小於 75 的判斷式轉換為整數 (True 轉 1, False 轉 0)
labeled["needs_support"] = (labeled["quiz_score"] < 75).astype(int)

print("各類別筆數：\n", labeled["needs_support"].value_counts().sort_index())

# 顯示標籤及分數，檢查低於 75 的定義
display(labeled[["student_id", "quiz_score", "needs_support"]].head(10))

# 自動化驗證
assert (labeled.loc[labeled["needs_support"] == 1, "quiz_score"] < 75).all(), "⚠️ 錯誤：有 needs_support 為 1 但成績高於或等於 75 分的學生！"
assert (labeled.loc[labeled["needs_support"] == 0, "quiz_score"] >= 75).all(), "⚠️ 錯誤：有 needs_support 為 0 但成績低於 75 分的學生！"
print("\n✅ 恭喜！自動化驗證完全通過，標籤建立無誤！")

### ✅ 作業 09-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [ ]:
assert labeled["needs_support"].equals((labeled["quiz_score"] < 75).astype(int))
assert len(labeled) == len(df)
print("檢查通過：分類門檻與筆數正確。")


### ✍️ 作業 09-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：＿＿「請協助我完成學生作業 09-P1 中的 9.2 建立教學標籤。目標是建立副本，再用條件判斷 quiz_score 是否小於 75 分並將其 astype(int) 轉成 1/0 的 needs_support 標籤。請提供 Python Pandas 程式碼，並附帶 assert 檢驗確保標籤建立無誤。」

＿＿
- 我補上的程式在做什麼：＿1.透過 df.copy() 複製原始資料表，確保後續分析的安全性。
2.運用 Pandas 向量化判斷式 (labeled["quiz_score"] < 75) 快速找出小考分數小於 75 分的學生，並呼叫 .astype(int) 將布林值（True/False）無縫轉換成二元分類標籤（1 代表需要教學輔助，0 代表不需要）。
3.分別印出 value_counts() 與 head(10) 來做分組筆數與局部資料的目視核對。＿＿＿
- 執行或驗證遇到的問題與修正：＿＿＿在執行前有特別注意要確保原始 df 資料表與 quiz_score 欄位已載入且沒有缺失值干擾，最終程式順利一次執行成功，並未遇到語法錯誤或斷言中斷。＿
- 檢查結果及我如何確認：＿＿＿＿第一重確認：藉由輸出統計，看見分類為 0 的有 46 筆，分類為 1 的有 14 筆，總計 60 筆完全脗合。
第二重確認：目視前 10 筆資料，發現學號 2（74分）、學號 3（72分）等低於 75 分的資料被精準標記為 1，其餘則為 0。
第三重確認：成功通過了兩個自動化斷言（assert）測試，印出「✅ 恭喜！自動化驗證完全通過，標籤建立無誤！」。


## 📝 學生作業 09-Q2｜AI 查詢與作答：標籤定義
**任務：**小考低於 75 分能否直接等同真實學生需要輔導？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請討論教學用標籤與實際輔導需求的差異，以及單次成績不應直接標籤學生的原因。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 09-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具：＿Colab Copilot (Gemini)
實際提示詞：我是 Python 與資料分析初學者。請討論教學用標籤與實際輔導需求的差異，以及單次成績不應直接標籤學生的原因。請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。＿＿＿
- AI 回答重點（請自行整理）：＿＿＿1.教學用標籤（模擬標籤）的局限性：在程式中利用特定門檻（例如 quiz_score < 75）建立的布林標籤 needs_support，在機器學習上僅屬於「二元分類門檻（Thresholding）」，這本質上只是對歷史數值進行「靜態特徵劃分」，不代表該名學生真實的學習動機、心理狀態或生活困境。
2.單次成績直接標籤學生的三大危機：
測驗誤差與隨機干擾：單次小考成績可能受到當天身體不適、考卷難度、隨機猜題（雜訊）等偶然因素干擾，不代表學生的長期實力。
自我實現預言與標籤效應（Pygmalion Effect）：若教師或系統過早將學生貼上「低成就」或「需輔導」標籤，可能會打擊學生自信心，導致其學習動機進一步退化。
漏判（偽陰性 / False Negative）與誤判（偽陽性 / False Positive）：有些人分數雖然高於 75 分，但可能是靠作弊、猜測、或是作息已經極度惡化但尚未反映在本次成績上（如螢幕時間突然暴增的人）；反之，有些人雖然低於 75 分，但只是偶然失常，不需要過度關懷。
3.實務輔導的多元評估：真正的教育現場，必須結合長期出席率、課堂參與度、導師質性訪談等多維度的指標，才能做出輔導判定，不能單憑單一量化分數。＿
- 我的理解與具體例子：＿＿數據分析上的 needs_support = 1 只能作為「考前初步篩選與預警的漏斗」，用來幫助老師縮小關懷的範圍，而絕對不能直接等同於真實的輔導診斷結果。標籤是冰冷的公式，但學生是活生生的人。＿＿
- 查證來源／方式及日期（無法查證請註明）：＿＿台灣教育部學生輔導諮商中心工作手冊、教育心理學「標籤理論 (Labeling Theory)」與「比馬龍效應 (Pygmalion Effect)」學術定義。＿＿
- 我檢查或修正 AI 回答的地方：＿＿＿AI 最初提供的驗證例子比較抽象，我將其修正並鎖定在我們本班級資料庫中的真實個案 —— 學號 9 號同學 的具體數據。他的小考分數達 80 分，但螢幕使用時間卻是極端的 15.0 小時。透過這個極端特徵與成績分佈的強烈對比，我具體論證了「偽陰性（漏判）」在實際輔導上的致命性，使得我的作答內容完美切合了 09-P1 的實作結果，具有極高的說服力！＿


## 9.3 任務規格與人工決策
填寫：預測對象、一列代表什麼、預測時間點、可取得欄位、目標、漏判與誤判的可能影響，以及後續人員如何使用結果。

**思考：**為什麼「學習表現不佳」不應直接等同「需要輔導」？一筆分數不足以代表個人狀況。

**提示詞：**請扮演助教，根據我填寫的規格提出三個需要澄清的問題，不要替我假設資料的取得時間。


## 延伸：檢查標籤門檻
定義標籤是一項業務決策，不能根據最終測試資料反覆選門檻。本示範只檢查既定的 75 分，並列出接近門檻的資料供人工理解。


In [ ]:
# @title 延伸：檢查 73–77 分的標籤與臨界點學生分佈
# 為了避免重複決策，本段程式僅列出 73-77 分臨界點學生，供人工理解與決策支援

# 1. 篩選出成績介於 73 至 77 分之間的學生資料
near_threshold = labeled[labeled["quiz_score"].between(73, 77)]

print("=== 接近 75 分門檻 (73-77分) 的學生標籤分佈 (按分數排序) ===")
display(near_threshold[["student_id", "quiz_score", "needs_support"]].sort_values("quiz_score"))

# 2. 統計總預警輔導人數 (1 類的數量)
print("--------------------------------------------------")
print("全班 60 人中，被標記為需要預警輔導 (1類) 的總人數：", int(labeled["needs_support"].sum()), "人")


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
